# Asura on a free GPU (Google Colab or Kaggle)

**One-time setup**
- **Colab:** Runtime → Change runtime type → *T4 GPU* → Save.
- **Kaggle:** Settings (right panel) → Accelerator → *GPU T4 x2*. Phone-verify your account first, or GPU stays greyed out. Then *Add Input → Upload* `asura.zip` as a dataset.

Then run the cells top to bottom. Results are zipped at the end for download.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# 2. Get the code: Kaggle reads the uploaded dataset, Colab asks you to upload asura.zip
import os, glob, zipfile
if os.path.exists("/kaggle/input"):
    z = (glob.glob("/kaggle/input/**/asura.zip", recursive=True) or [None])[0]
    if z: zipfile.ZipFile(z).extractall("/kaggle/working")
    else:  # Kaggle auto-extracts zips into a folder; copy it
        src = glob.glob("/kaggle/input/**/asura/model.py", recursive=True)[0]
        !cp -r {os.path.dirname(os.path.dirname(src))} /kaggle/working/asura_src
    root = "/kaggle/working/asura_src" if not z else "/kaggle/working"
else:
    from google.colab import files
    up = files.upload()                      # pick asura.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content")
    root = "/content"
os.chdir(root); print("working in", os.getcwd()); print(os.listdir("."))

In [ ]:
# 3. Smoke test (~1-2 min on GPU vs ~9 min on the laptop CPU)
!python -W ignore -m asura.train experiments/configs/smoke.json

In [ ]:
# 4. Paper experiments (each picks up where it left off if re-run)
!PYTHONIOENCODING=utf-8 python -W ignore -u experiments/run.py experiments/configs/clean_data_paradox.json
!python -W ignore -u experiments/faithfulness/u3.py --seeds 3 --steps 1500
!python -W ignore -u experiments/trust_opt/u1.py --seeds 2
!python -W ignore experiments/imported_algo/sim.py --seeds 5

### 5. Scale up (only possible on GPU)
More seeds = tighter confidence intervals. Bigger models = closer to real LMs. Examples:

In [ ]:
# 5 seeds instead of 3, for the Clean Data paper
import json
s = json.load(open("experiments/configs/clean_data_paradox.json")); s["seeds"] = [0,1,2,3,4]; s["name"] += "_5seeds"
json.dump(s, open("experiments/configs/clean_data_paradox_5seeds.json", "w"))
!PYTHONIOENCODING=utf-8 python -W ignore -u experiments/run.py experiments/configs/clean_data_paradox_5seeds.json

# A ~10M-parameter Asura (30x the laptop model)
!python -W ignore -m asura.train experiments/configs/smoke.json --set name=smoke_10M model.d_model=384 model.n_layers=6 model.n_heads=6 train.steps=3000

In [ ]:
# 6. Download results (Colab: downloads; Kaggle: appears under Output)
import shutil; shutil.make_archive("asura_runs", "zip", "runs")
try:
    from google.colab import files; files.download("asura_runs.zip")
except ImportError:
    print("Kaggle: find asura_runs.zip in the Output tab")